In [ ]:
"""
RAG with Qdrant + OpenAI (embeddings + chat).
"""

import os
import uuid
from pathlib import Path

from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

In [ ]:
from pathlib import Path

from dotenv import load_dotenv

_repo_root = Path.cwd()
if not (_repo_root / "requirements.txt").is_file():
    _repo_root = _repo_root.parent
load_dotenv(_repo_root / ".env", override=True)


def _require_env(name: str) -> str:
    """Return a required env var or abort if missing or still a placeholder."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise EnvironmentError(
            f"Missing required environment variable {name!r}. "
            "Copy .env.example to .env at the ask-it repo root and set a real value "
            "(see 04_starter_kit/README.md)."
        )
    if "<" in value or "***" in value:
        raise EnvironmentError(
            f"Environment variable {name!r} looks like a placeholder. "
            "Update ask-it/.env with real credentials."
        )
    return value


# Hosted Qdrant + Vayu Model as a Service — required secrets (no inline defaults)
QDRANT_URL = _require_env("QDRANT_URL")
QDRANT_API_KEY = _require_env("QDRANT_API_KEY")
LLM_OPENAI_API_KEY = _require_env("LLM_OPENAI_API_KEY")
EMBEDDING_OPENAI_API_KEY = _require_env("EMBEDDING_OPENAI_API_KEY")
OPENAI_BASE_URL = _require_env("OPENAI_BASE_URL")

# Models — override via .env if needed
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "").strip() or "Qwen/Qwen3-Embedding-8B"
CHAT_MODEL = os.environ.get("CHAT_MODEL", "").strip() or "openai/gpt-oss-120b"
COLLECTION_NAME = os.environ.get("COLLECTION_NAME", "").strip() or "knowledge_base_rag"

# Local knowledge: all readable text files under this folder (recursive)
DOCS_DIR = "docs/"
TEXT_EXTENSIONS = {".md", ".markdown", ".txt", ".html", ".htm", ".rst", ".csv"}
MAX_CHUNK_CHARS = 1500
CHUNK_OVERLAP = 120
EMBED_BATCH_SIZE = 32

In [ ]:
# Use OPENAI_BASE_URL from the config cell — never hardcode tenant URLs in source.
embedding_openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=EMBEDDING_OPENAI_API_KEY)
llm_openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=LLM_OPENAI_API_KEY)

In [ ]:
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)

In [ ]:
# write code to fetch embedding model size
response = embedding_openai_client.embeddings.create(
    input=["Hello world!"],
    model=EMBEDDING_MODEL
)
embedding = response.data[0].embedding
embedding_size = len(embedding)
print(f"Embedding size: {embedding_size}")

In [ ]:
VECTOR_SIZE = embedding_size

def ensure_collection(qdrant_client, recreate: bool = False) -> None:
    """Create Qdrant collection sized for OpenAI text-embedding-3-small."""
    existing = {c.name for c in qdrant_client.get_collections().collections}
    if COLLECTION_NAME in existing:
        if recreate:
            qdrant_client.delete_collection(COLLECTION_NAME)
        else:
            print(f"Collection {COLLECTION_NAME!r} already exists; skip create.")
            return
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    print(f"Created collection {COLLECTION_NAME!r} (dim={VECTOR_SIZE}).")

def local_db(collection_name=COLLECTION_NAME, vector_size=VECTOR_SIZE):
    """
    Create and return a local Qdrant collection for embeddings.
    If the collection exists, it will not be recreated.
    """
    import tempfile
    from qdrant_client import QdrantClient
    from qdrant_client.models import VectorParams, Distance

    # Use a temporary directory for the local Qdrant storage (or set your path)
    storage_path = os.curdir
    client = QdrantClient(path=storage_path)
    existing = {c.name for c in client.get_collections().collections}
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print(f"Created local collection {collection_name!r} (dim={vector_size}) at {storage_path}")
    else:
        print(f"Local collection {collection_name!r} already exists at {storage_path}")
    return client

def hosted_instance(collection_name=COLLECTION_NAME, vector_size=VECTOR_SIZE):
    """
    Connect to a hosted Qdrant instance and ensure the collection exists.
    If the collection does not exist, it will be created.
    """
    from qdrant_client import QdrantClient
    from qdrant_client.models import VectorParams, Distance

    # Use env-configured URL and API key
    QDRANT_URL = os.environ["QDRANT_URL"]
    QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]
    client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        port=443
    )
    existing = {c.name for c in client.get_collections().collections}
    if collection_name not in existing:
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print(f"Created hosted collection {collection_name!r} (dim={vector_size}) at {QDRANT_URL}")
    else:
        print(f"Hosted collection {collection_name!r} already exists at {QDRANT_URL}")
    return client

# Usage: Replace qdrant with local Qdrant client for now
# qdrant = local_db()
# To switch to hosted instance, comment above line and uncomment below line:
qdrant = hosted_instance()

# To ensure your collection with optional recreation, use:
# ensure_collection(qdrant)           # creates the collection if it doesn't exist
ensure_collection(qdrant, recreate=True)  # recreates (deletes and creates) the collection

In [ ]:
def _split_oversized(text: str, max_chars: int, overlap: int) -> list[str]:
    if len(text) <= max_chars:
        return [text] if text.strip() else []
    parts: list[str] = []
    start = 0
    while start < len(text):
        parts.append(text[start : start + max_chars])
        start += max_chars - overlap
    return parts


def chunk_file_text(text: str, max_chars: int, overlap: int) -> list[str]:
    """Split file content into overlapping character windows after paragraph merge."""
    text = text.strip()
    if not text:
        return []
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: list[str] = []
    buf = ""
    for para in paragraphs:
        if len(buf) + len(para) + 2 <= max_chars:
            buf = f"{buf}\n\n{para}".strip() if buf else para
        else:
            if buf:
                chunks.extend(_split_oversized(buf, max_chars, overlap))
            if len(para) <= max_chars:
                buf = para
            else:
                chunks.extend(_split_oversized(para, max_chars, overlap))
                buf = ""
    if buf:
        chunks.extend(_split_oversized(buf, max_chars, overlap))
    return [c for c in chunks if c.strip()]


def iter_doc_files(root: Path):
    if not root.is_dir():
        return
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in TEXT_EXTENSIONS:
            yield path


def load_chunks_from_docs(
    docs_dir: Path,
    *,
    max_chunk_chars: int = MAX_CHUNK_CHARS,
    overlap: int = CHUNK_OVERLAP,
) -> list[dict]:
    """Read every text file under docs_dir; each chunk includes source path for citations."""
    root = docs_dir
    out: list[dict] = []
    if not root.exists():
        root.mkdir(parents=True, exist_ok=True)
    if not root.is_dir():
        print(f"Docs path is not a folder: {root}")
        return out

    for file_path in iter_doc_files(root):
        try:
            raw = file_path.read_text(encoding="utf-8", errors="replace")
        except OSError as e:
            print(f"Skip {file_path}: {e}")
            continue
        rel = str(file_path.relative_to(root))
        for chunk in chunk_file_text(raw, max_chunk_chars, overlap):
            out.append({"text": chunk, "source": rel})
    return out


def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed with OpenAI in batches (order preserved)."""
    if not texts:
        return []
    all_vectors: list[list[float]] = []
    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i : i + EMBED_BATCH_SIZE]
        res = embedding_openai_client.embeddings.create(input=batch, model=EMBEDDING_MODEL)
        all_vectors.extend(d.embedding for d in res.data)
    return all_vectors


def upsert_chunks(chunks: list[dict]) -> None:
    """Each chunk dict: text, source (path under docs folder)."""
    if not chunks:
        print("No chunks to index; add files under", DOCS_DIR)
        return
    texts = [c["text"] for c in chunks]
    vectors = embed_texts(texts)
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=v,
            payload={"text": c["text"], "source": c["source"]},
        )
        for v, c in zip(vectors, chunks)
    ]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
    print(f"Indexed {len(points)} chunks from {DOCS_DIR} into {COLLECTION_NAME!r}.")


def retrieve(question: str, limit: int = 4):
    """Vector search: qdrant-client 1.10+ uses query_points; older clients used search."""
    qvec = embed_texts([question])[0]
    if hasattr(qdrant, "query_points"):
        res = qdrant.query_points(
            collection_name=COLLECTION_NAME,
            query=qvec,
            limit=limit,
            with_payload=True,
        )
        return res.points
    return qdrant.search(
        collection_name=COLLECTION_NAME,
        query_vector=qvec,
        limit=limit,
    )


def rag_answer(question: str, top_k: int = 4) -> str:
    hits = retrieve(question, limit=top_k)
    blocks = []
    for h in hits:
        if not h.payload:
            continue
        src = h.payload.get("source", "")
        body = h.payload.get("text", "")
        blocks.append(f"[{src}]\n{body}" if src else body)
    context = "\n\n---\n\n".join(blocks)

    system = (
        "You are a helpful assistant. Answer using only the provided context. "
        "If the context is insufficient, say you do not know. "
        "Mention which source file the answer came from when possible."
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"

    completion = llm_openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.2,
    )
    return completion.choices[0].message.content or ""


# --- Index everything under DOCS_DIR (set ensure_collection(recreate=True) in prior cell to wipe old vectors) ---
from pathlib import Path

DOCS_DIR = Path("../01_dataset/docs")  # Hardcoded docs path

chunks = load_chunks_from_docs(DOCS_DIR)
upsert_chunks(chunks)

demo_q = "What are the remainders for the field trip."
answer = rag_answer(demo_q)
print("Q:", demo_q)
print("A:", answer)